In [6]:
import tkinter as tk
from tkinter import messagebox
import pymysql
import re
import webbrowser
import os
from datetime import datetime

class DictionaryApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Python 수업 용어 사전")
        self.root.geometry("480x800")
        
        # 설정 및 폰트
        self.config_file = "db_config.txt"
        self.font_main = ("Malgun Gothic", 10)
        self.font_bold = ("Malgun Gothic", 11, "bold")
        self.font_title = ("Malgun Gothic", 15, "bold")
        self.font_info = ("Malgun Gothic", 9, "italic")
        
        self.db_host = "localhost"
        self.connection = None
        self.current_user = ""
        
        # 검색 히스토리 관리
        self.history = []
        self.history_index = -1
        
        # 자동 로그인 확인
        self.check_auto_login()

    def check_auto_login(self):
        """저장된 설정 파일이 있는지 확인"""
        if os.path.exists(self.config_file):
            try:
                with open(self.config_file, "r", encoding="utf-8") as f:
                    lines = f.readlines()
                    saved_id = lines[0].strip()
                    saved_pw = lines[1].strip()
                
                # 저장된 정보로 접속 시도
                if self.perform_login(saved_id, saved_pw, silent=True):
                    return # 성공 시 로그인 창 띄우지 않음
            except:
                pass
        
        # 저장된 정보가 없거나 접속 실패 시 로그인 화면 표시
        self.show_login_ui()

    def show_login_ui(self):
        """로그인 화면 구성"""
        self.login_frame = tk.Frame(self.root)
        self.login_frame.pack(pady=50)
        
        tk.Label(self.login_frame, text="🐍 Python 수업 용어 사전", font=self.font_title).pack(pady=20)
        tk.Label(self.login_frame, text="리눅스 계정 정보를 입력하세요", font=self.font_info, fg="gray").pack()
        
        tk.Label(self.login_frame, text="Linux ID:", font=self.font_main).pack(pady=(10, 0))
        self.ent_user = tk.Entry(self.login_frame, font=self.font_main)
        self.ent_user.pack(pady=5)
        self.ent_user.bind('<Return>', lambda event: self.ent_pw.focus())
        
        tk.Label(self.login_frame, text="Password:", font=self.font_main).pack()
        self.ent_pw = tk.Entry(self.login_frame, show="*", font=self.font_main)
        self.ent_pw.pack(pady=5)
        self.ent_pw.bind('<Return>', lambda event: self.btn_login_click())
        
        tk.Button(self.login_frame, text="데이터베이스 접속", font=self.font_bold,
                  command=self.btn_login_click, bg="#2196F3", fg="white", width=20).pack(pady=20)

    def btn_login_click(self):
        """로그인 버튼 클릭 시 처리"""
        user_id = self.ent_user.get().strip()
        user_pw = self.ent_pw.get().strip()
        
        if self.perform_login(user_id, user_pw):
            # 성공 시 파일에 저장
            with open(self.config_file, "w", encoding="utf-8") as f:
                f.write(f"{user_id}\n{user_pw}")

    def perform_login(self, user_id, user_pw, silent=False):
        """실제 DB 접속 수행"""
        try:
            self.connection = pymysql.connect(
                host=self.db_host,
                user=user_id,
                password=user_pw,
                db='class_dict_db',
                charset='utf8mb4',
                cursorclass=pymysql.cursors.DictCursor,
                autocommit=True
            )
            self.current_user = user_id
            if not silent:
                messagebox.showinfo("성공", f"계정 '{user_id}' 인증 성공!")
            self.show_search_ui()
            return True
        except Exception as e:
            if not silent:
                messagebox.showerror("오류", "계정 정보가 올바르지 않거나 접속이 거부되었습니다.")
            return False

    def show_search_ui(self):
        if hasattr(self, 'login_frame'):
            self.login_frame.destroy()
        
        self.search_frame = tk.Frame(self.root)
        self.search_frame.pack(pady=20, fill="both", expand=True)
        
        # 로그아웃(설정삭제) 버튼 추가
        top_frame = tk.Frame(self.search_frame)
        top_frame.pack(fill="x", padx=10)
        tk.Button(top_frame, text="로그아웃", font=self.font_info, command=self.logout).pack(side="left")
        tk.Label(top_frame, text=f"접속 계정: {self.current_user}", font=self.font_info, fg="gray").pack(side="right")
        
        tk.Label(self.search_frame, text="검색할 용어를 입력하세요", font=self.font_bold).pack(pady=10)
        
        self.ent_search = tk.Entry(self.search_frame, width=40, font=self.font_main)
        self.ent_search.pack(pady=5)
        self.ent_search.focus_set()
        
        self.list_suggestion = tk.Listbox(self.search_frame, width=40, height=0, font=self.font_main)
        self.list_suggestion.pack()
        self.list_suggestion.pack_forget() 
        
        self.ent_search.bind('<KeyRelease>', self.on_key_release)
        self.ent_search.bind('<Return>', lambda e: self.search_word())
        self.list_suggestion.bind("<<ListboxSelect>>", self.select_from_list)
        
        btn_frame = tk.Frame(self.search_frame)
        btn_frame.pack(pady=5)
        tk.Button(btn_frame, text="단어 검색", font=self.font_bold, command=self.search_word, width=12).pack(side="left", padx=5)
        tk.Button(btn_frame, text="이전 검색", font=self.font_main, command=self.go_back_history, bg="#f9f9f9", width=12).pack(side="left", padx=5)
        
        self.txt_result = tk.Text(self.search_frame, height=20, width=55, font=self.font_main, padx=10, pady=10)
        self.txt_result.pack(pady=10, padx=20)
        self.txt_result.tag_config("link", foreground="blue", underline=True)
        self.txt_result.tag_config("info", foreground="gray", font=self.font_info)

        tk.Button(self.search_frame, text="+ 새 용어 등록하기", font=self.font_main,
                  command=self.open_add_window, bg="#eeeeee").pack(pady=10)

    def logout(self):
        """저장된 비밀번호 삭제 후 재시작"""
        if os.path.exists(self.config_file):
            os.remove(self.config_file)
        messagebox.showinfo("로그아웃", "자동 로그인 정보가 삭제되었습니다. 프로그램을 재시작합니다.")
        self.root.destroy()

    def on_key_release(self, event):
        if event.keysym == "Down" and self.list_suggestion.winfo_viewable():
            self.list_suggestion.focus_set()
            self.list_suggestion.selection_set(0)
            return
        search_text = self.ent_search.get().strip()
        if not search_text:
            self.list_suggestion.pack_forget()
            return
        try:
            self.connection.ping(reconnect=True)
            with self.connection.cursor() as cursor:
                sql = "SELECT word FROM terms WHERE word LIKE %s LIMIT 5"
                cursor.execute(sql, (f"%{search_text}%",))
                results = cursor.fetchall()
                if results:
                    self.list_suggestion.delete(0, tk.END)
                    for item in results: self.list_suggestion.insert(tk.END, item['word'])
                    self.list_suggestion.config(height=len(results))
                    self.list_suggestion.pack(after=self.ent_search)
                else: self.list_suggestion.pack_forget()
        except: self.list_suggestion.pack_forget()

    def select_from_list(self, event):
        if not self.list_suggestion.curselection(): return
        selected_word = self.list_suggestion.get(self.list_suggestion.curselection())
        self.ent_search.delete(0, tk.END)
        self.ent_search.insert(0, selected_word)
        self.list_suggestion.pack_forget()
        self.search_word()

    def search_word(self):
        word = self.ent_search.get().strip()
        self.list_suggestion.pack_forget()
        if not word: return
        try:
            with self.connection.cursor() as cursor:
                sql = "SELECT * FROM terms WHERE word = %s"
                cursor.execute(sql, (word,))
                result = cursor.fetchone()
                self.txt_result.delete(1.0, tk.END)
                if result:
                    content = result.get('definition', '')
                    author = result.get('author', '알 수 없음')
                    date = result.get('created_at', '정보 없음')
                    self.display_with_links(self.txt_result, f"[{word}]\n\n{content}")
                    self.txt_result.insert(tk.END, "\n\n" + "-"*40 + "\n", "info")
                    self.txt_result.insert(tk.END, f"✍ 작성자: {author}  |  📅 일자: {date}", "info")
                    if not self.history or self.history[-1] != word:
                        self.history.append(word)
                        if len(self.history) > 30: self.history.pop(0)
                    self.history_index = len(self.history) - 1
                else: self.txt_result.insert(tk.END, "검색 결과가 없습니다.")
        except Exception as e: messagebox.showerror("오류", f"조회 중 오류 발생: {e}")

    def display_with_links(self, widget, text):
        url_pattern = r'(https?://[^\s]+)'
        parts = re.split(url_pattern, text)
        for part in parts:
            if re.match(url_pattern, part):
                tag_name = f"link-{part}"
                widget.insert(tk.END, part, ("link", tag_name))
                widget.tag_bind(tag_name, "<Button-1>", lambda e, url=part: webbrowser.open(url))
            else: widget.insert(tk.END, part)

    def go_back_history(self):
        if len(self.history) <= 1 or self.history_index <= 0:
            messagebox.showinfo("알림", "이전 검색 기록이 없습니다.")
            return
        self.history_index -= 1
        prev_word = self.history[self.history_index]
        self.ent_search.delete(0, tk.END)
        self.ent_search.insert(0, prev_word)
        self.search_word()

    def open_add_window(self):
        self.add_win = tk.Toplevel(self.root)
        self.add_win.title("새 용어 등록")
        self.add_win.geometry("400x550")
        tk.Label(self.add_win, text="📖 새 용어 추가", font=self.font_bold).pack(pady=15)
        tk.Label(self.add_win, text="단어명:", font=self.font_main).pack()
        self.ent_new_word = tk.Entry(self.add_win, width=35, font=self.font_main)
        self.ent_new_word.pack(pady=5); self.ent_new_word.focus_set()
        tk.Label(self.add_win, text="단어 내용(뜻):", font=self.font_main).pack(pady=5)
        self.txt_new_def = tk.Text(self.add_win, height=10, width=45, font=self.font_main, padx=5, pady=5)
        self.txt_new_def.pack(pady=5)
        tk.Label(self.add_win, text=f"기록될 작성자: {self.current_user}", font=self.font_info, fg="blue").pack(pady=5)
        tk.Button(self.add_win, text="DB 기록 완료", font=self.font_bold, command=self.save_to_db, bg="#4CAF50", fg="white", width=20).pack(pady=15)

    def save_to_db(self):
        new_word = self.ent_new_word.get().strip()
        new_def = self.txt_new_def.get(1.0, tk.END).strip()
        if not new_word or not new_def:
            messagebox.showwarning("누락", "모든 항목을 입력하세요.")
            return
        try:
            with self.connection.cursor() as cursor:
                check_sql = "SELECT word FROM terms WHERE word = %s"
                cursor.execute(check_sql, (new_word,))
                if cursor.fetchone():
                    messagebox.showwarning("중복 오류", f"'{new_word}'은(는) 이미 등록된 단어입니다.")
                    return
                sql = "INSERT INTO terms (word, definition, author, created_at) VALUES (%s, %s, %s, NOW())"
                cursor.execute(sql, (new_word, new_def, self.current_user))
                self.connection.commit()
            messagebox.showinfo("성공", f"'{new_word}' 등록 성공!")
            self.add_win.destroy()
        except Exception as e: messagebox.showerror("저장 오류", f"DB 기록 실패: {e}")

if __name__ == "__main__":
    root = tk.Tk()
    app = DictionaryApp(root)
    root.mainloop()
